In [ ]:
from pathlib import Path

# Locate the archive from the repository root or this notebook's directory.
def _find_archive_root():
    for directory in (Path.cwd(), *Path.cwd().parents):
        for candidate in (directory, directory / "M0" / "training" / "benchmark_archive"):
            if (candidate / "inputs" / "original_m0_tm.csv").is_file():
                return candidate
    raise FileNotFoundError("Run this notebook from the repository or benchmark_archive directory.")

ARCHIVE_DIR = _find_archive_root()
ARCHIVE_INPUT_DIR = ARCHIVE_DIR / "inputs"
ARCHIVE_OUTPUT_DIR = ARCHIVE_DIR / "notebook_outputs"
ARCHIVE_OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
import optuna
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split

# =======================
# Read the complete dataset.
# =======================
data_df = pd.read_csv(
    str(ARCHIVE_INPUT_DIR / "original_m0_single_peak.csv")
)

# =======================
# Prepare features and targets.
# =======================
X = data_df.iloc[:, 1:-1]   # Feature columns
y = data_df.iloc[:, -1]     # Target column

# =======================
# Split into 80% training and 20% test data.
# =======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =======================
# Define the optimization objective.
# =======================
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 1000)
    max_depth = trial.suggest_int('max_depth', 10, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        max_features=max_features,
        random_state=42
    )

    auc_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='roc_auc')
    return auc_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

print("Best hyperparameters:", study.best_params)
print("Best AUC-ROC:", study.best_value)

best_rf = RandomForestClassifier(**study.best_params, random_state=42)
best_rf.fit(X_train, y_train)

y_pred = best_rf.predict(X_test)
test_auc = roc_auc_score(y_test, best_rf.predict_proba(X_test)[:, 1])
test_accuracy = accuracy_score(y_test, y_pred)
print("Test AUC-ROC:", test_auc)
print("Test Accuracy:", test_accuracy)

import joblib
joblib.dump(best_rf, "best_random_forest.pkl")


[I 2026-03-03 17:16:19,495] A new study created in memory with name: no-name-130463b1-39bc-4def-91ca-91830abc7096
[I 2026-03-03 17:16:21,440] Trial 0 finished with value: 0.8454166666666666 and parameters: {'n_estimators': 754, 'max_depth': 47, 'min_samples_split': 6, 'min_samples_leaf': 9, 'bootstrap': True, 'max_features': 'log2'}. Best is trial 0 with value: 0.8454166666666666.
[I 2026-03-03 17:16:23,619] Trial 1 finished with value: 0.7991406249999999 and parameters: {'n_estimators': 717, 'max_depth': 23, 'min_samples_split': 19, 'min_samples_leaf': 4, 'bootstrap': False, 'max_features': None}. Best is trial 0 with value: 0.8454166666666666.
[I 2026-03-03 17:16:24,362] Trial 2 finished with value: 0.8250260416666666 and parameters: {'n_estimators': 280, 'max_depth': 12, 'min_samples_split': 4, 'min_samples_leaf': 10, 'bootstrap': False, 'max_features': None}. Best is trial 0 with value: 0.8454166666666666.
[I 2026-03-03 17:16:24,911] Trial 3 finished with value: 0.8514843750000001 

Best hyperparameters: {'n_estimators': 1000, 'max_depth': 47, 'min_samples_split': 2, 'min_samples_leaf': 1, 'bootstrap': True, 'max_features': None}
Best AUC-ROC: 0.891796875
Test AUC-ROC: 0.8823529411764707
Test Accuracy: 0.85


['best_random_forest.pkl']